<a href="https://colab.research.google.com/github/musorsey-web/lab/blob/main/lab_02_(5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 2
## Колоночное хранение, партиционирование и бенчмаркинг Parquet

**Дисциплина:** «Методы обработки больших данных»
**Направление:** 44.03.05 Педагогическое образование
**Профиль:** «Информатика и дополнительное образование (робототехника)»
**Этап конвейера:** `Collect → Store → Process`
**Среда:** Google Colab · PySpark 4.2.0 · `local[*]` · Parquet · Snappy / ZSTD / GZIP

---

### Цель работы

Освоить колоночное хранение телеметрии в формате Parquet, физическое партиционирование набора
и экспериментальное сравнение объёма и времени чтения CSV и Parquet. Понять, за счёт чего
возникает выигрыш и когда он исчезает.

### Что нужно сдать

1. Этот ноутбук с выполненными заданиями (`.ipynb`, все ячейки выполнены сверху вниз).
2. Ссылку на ноутбук в GitHub или Colab.
3. Заполненную форму **TEACH CARD** в конце ноутбука.

### Критерии оценки — ровно 10 баллов

| Часть | Содержание | Баллы |
|---|---|---|
| 1 | Три физические версии набора: CSV, Parquet, partitioned Parquet | 2 |
| 2 | Размеры, коэффициент сжатия, влияние энтропии и выбора кодека | 2 |
| 3 | Воспроизводимый бенчмарк чтения: ≥ 3 повтора, медиана, честное сравнение | 2 |
| 4 | Column pruning, partition pruning, проблема мелких файлов | 2 |
| 5 | Заполненная форма TEACH CARD | 2 |

> **Правила выполнения.** Не ищите готовые решения — пользуйтесь `help()` и документацией Spark.
> Ячейки с `assert` — самопроверка: если проверка прошла, задание засчитано. Числовые значения
> времени зависят от runtime Colab, поэтому оценивается методика измерения, а не абсолютные цифры.

### Источники

* Balapriya C. `data-science-tutorials` — <https://github.com/balapriyac/data-science-tutorials>
  (`pyspark/pyspark_write_parquet.ipynb`)
* Piotr Szul. `spark-tutorial` — <https://github.com/piotrszul/spark-tutorial>
* Apache Parquet — <https://parquet.apache.org/docs/>
* Spark. Parquet Files — <https://spark.apache.org/docs/latest/sql-data-sources-parquet.html>

---
## Задание 0. Подготовка среды и данных

### Минимальные теоретические сведения

**Колоночное хранение.** Parquet размещает рядом значения одной колонки, а не одной записи.
Отсюда два следствия: однородные данные сжимаются в разы лучше, а запрос, которому нужны две
колонки из десяти, читает только эти две — это **column pruning**.

**Партиционирование.** `partitionBy("robot_id")` раскладывает данные по каталогам вида
`robot_id=robot-7/`. Запрос с фильтром по этой колонке пропускает лишние каталоги целиком —
это **partition pruning**. Значение ключа при этом физически не хранится в файлах: оно
закодировано в имени каталога.

**Метрики работы:**

коэффициент уменьшения объёма  $C = S_{CSV} / S_{Parquet}$

ускорение чтения  $Speedup = T_{CSV} / T_{Parquet}$

Результаты зависят от runtime Colab и файлового кэша, поэтому каждое измерение повторяется
не менее трёх раз, а итогом берётся **медиана** (она устойчивее среднего к единичному выбросу
при первом «холодном» чтении).

In [1]:
!pip -q install "pyspark==4.0.0" pandas pyarrow

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import shutil
import time
import statistics
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab02_ParquetBenchmark")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version:", spark.version)

ROOT = Path("/content/lab02_storage")
if not Path("/content").exists():          # запуск вне Colab
    ROOT = Path("./lab02_storage")

CSV_DIR         = ROOT / "csv"
PARQUET_DIR     = ROOT / "parquet"
PARTITIONED_DIR = ROOT / "parquet_partitioned"

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
print("Каталог хранения:", ROOT)

Spark version: 4.0.0
Каталог хранения: /content/lab02_storage


### Генерация телеметрии

Набор порождается детерминированно, внешний датасет не нужен. Обратите внимание на колонку
`payload`: это SHA-256 от идентификатора, то есть **высокоэнтропийная** строка без повторов.
Она добавлена намеренно — в части 2 мы посмотрим, что она делает со сжатием.

DataFrame кэшируется: иначе Spark пересчитает его заново при каждой из трёх записей.

In [3]:
N = 300_000

df = (
    spark.range(N)
    .withColumn("robot_id",    F.concat(F.lit("robot-"), (F.col("id") % 12).cast("string")))
    .withColumn("sensor",      F.when(F.col("id") % 2 == 0, F.lit("imu")).otherwise(F.lit("motor")))
    .withColumn("ts_ms",       (F.lit(1_720_000_000_000) + F.col("id") * 100).cast("long"))
    .withColumn("temperature", F.lit(40.0) + (F.col("id") % 150) * 0.08)
    .withColumn("vibration",   F.abs(F.sin(F.col("id") / 15.0)) + (F.col("id") % 11) * 0.002)
    .withColumn("payload",     F.sha2(F.col("id").cast("string"), 256))
    .drop("id")
    .cache()
)

print("Строк:", df.count())
df.printSchema()
df.show(3, truncate=False)

Строк: 300000
root
 |-- robot_id: string (nullable = true)
 |-- sensor: string (nullable = false)
 |-- ts_ms: long (nullable = false)
 |-- temperature: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- payload: string (nullable = true)

+--------+------+-------------+-----------+-------------------+----------------------------------------------------------------+
|robot_id|sensor|ts_ms        |temperature|vibration          |payload                                                         |
+--------+------+-------------+-----------+-------------------+----------------------------------------------------------------+
|robot-0 |imu   |1720000000000|40.0       |0.0                |5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9|
|robot-1 |motor |1720000000100|40.08      |0.068617294923393  |6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b|
|robot-2 |imu   |1720000000200|40.16      |0.13693862262231407|d4735e3a265e16eee03f59718b9b5d0301

---
# Часть 1. Три физические версии набора (2 балла)

Один и тот же DataFrame нужно записать тремя способами и дальше сравнивать их между собой.

## Упражнение 1.1. CSV

Запишите `df` в CSV в каталог `CSV_DIR`. Режим записи — `overwrite`, заголовок включён
(`option("header", True)`).

In [4]:
df.write.mode("overwrite").option("header", True).csv(str(CSV_DIR))

files = sorted(p.name for p in CSV_DIR.iterdir())
print("Файлов в каталоге CSV:", len(files))
print(files[:5])
assert CSV_DIR.exists() and any(p.suffix == ".csv" for p in CSV_DIR.iterdir())

Файлов в каталоге CSV: 6
['._SUCCESS.crc', '.part-00000-231b2136-eed1-4ac6-85a0-ff6b2863be14-c000.csv.crc', '.part-00001-231b2136-eed1-4ac6-85a0-ff6b2863be14-c000.csv.crc', '_SUCCESS', 'part-00000-231b2136-eed1-4ac6-85a0-ff6b2863be14-c000.csv']


> **Обратите внимание.** Spark пишет не один файл, а каталог: по одному файлу `part-*` на каждую
> partition плюс маркер `_SUCCESS`. Так устроены все распределённые форматы — единый файл нельзя
> писать параллельно с нескольких узлов.

## Упражнение 1.2. Parquet со сжатием Snappy

Запишите тот же `df` в Parquet в каталог `PARQUET_DIR`, явно указав кодек
`option("compression", "snappy")`.

In [5]:
df.write.mode("overwrite").option("compression", "snappy").parquet(str(PARQUET_DIR))

parts = [p.name for p in PARQUET_DIR.iterdir() if p.suffix == ".parquet"]
print("Файлов parquet:", len(parts))
print(parts[:3])
assert len(parts) > 0
assert all("snappy" in p for p in parts), "В имени файла должен быть виден кодек"

Файлов parquet: 2
['part-00000-508a1208-6a63-43f0-a5fd-47e74d8f3e48-c000.snappy.parquet', 'part-00001-508a1208-6a63-43f0-a5fd-47e74d8f3e48-c000.snappy.parquet']


## Упражнение 1.3. Partitioned Parquet

Запишите набор в `PARTITIONED_DIR` с физическим партиционированием по `robot_id`
(`partitionBy`). Затем выведите список созданных каталогов и проверьте, что их ровно 12 —
по числу роботов.

In [6]:
df.write.mode("overwrite").partitionBy("robot_id").parquet(str(PARTITIONED_DIR))

part_dirs = sorted(p.name for p in PARTITIONED_DIR.iterdir() if p.is_dir())
print("Каталогов:", len(part_dirs))
print(part_dirs[:4])

assert len(part_dirs) == 12
assert all(d.startswith("robot_id=") for d in part_dirs)
assert all(d.startswith("robot_id=") for d in part_dirs)

Каталогов: 12
['robot_id=robot-0', 'robot_id=robot-1', 'robot_id=robot-10', 'robot_id=robot-11']


## Упражнение 1.4

Прочитайте один файл из каталога `robot_id=robot-7` напрямую (`spark.read.parquet` по полному
пути к каталогу партиции) и посмотрите его схему. Есть ли в ней колонка `robot_id`?
Ответьте письменно, почему.

In [7]:
one_partition = spark.read.parquet(str(PARTITIONED_DIR / "robot_id=robot-7"))
one_partition.printSchema()

ANSWER_1_4 = """
Колонки robot_id в схеме нет. При партиционировании Spark не хранит значение ключа внутри файлов:
оно закодировано в имени каталога robot_id=robot-7. Это само по себе экономит место — строка
"robot-7" не повторяется 25 000 раз, а записана один раз в имени каталога.

Когда мы читаем весь каталог PARTITIONED_DIR целиком, Spark восстанавливает колонку robot_id
из имён подкаталогов (partition discovery), и она снова появляется в схеме. Но если читать
отдельную партицию напрямую, восстанавливать не из чего — контекст потерян.
"""
print(ANSWER_1_4)

assert "robot_id" not in one_partition.columns

root
 |-- sensor: string (nullable = true)
 |-- ts_ms: long (nullable = true)
 |-- temperature: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- payload: string (nullable = true)


Колонки robot_id в схеме нет. При партиционировании Spark не хранит значение ключа внутри файлов:
оно закодировано в имени каталога robot_id=robot-7. Это само по себе экономит место — строка
"robot-7" не повторяется 25 000 раз, а записана один раз в имени каталога.

Когда мы читаем весь каталог PARTITIONED_DIR целиком, Spark восстанавливает колонку robot_id
из имён подкаталогов (partition discovery), и она снова появляется в схеме. Но если читать
отдельную партицию напрямую, восстанавливать не из чего — контекст потерян.



---
# Часть 2. Объём хранения (2 балла)

## Упражнение 2.1. Размеры и коэффициент сжатия

Напишите функцию `dir_size(path)`, возвращающую суммарный размер всех файлов каталога в байтах
(каталог вложенный, поэтому нужен рекурсивный обход — подсказка: `Path.rglob`).

Выведите таблицу размеров трёх версий и рассчитайте коэффициент
$C = S_{CSV} / S_{Parquet}$.

In [8]:
def dir_size(path: Path) -> int:
    """Суммарный размер всех файлов внутри каталога, включая вложенные."""
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


sizes = {
    "CSV":                 dir_size(CSV_DIR),
    "Parquet":             dir_size(PARQUET_DIR),
    "Partitioned Parquet": dir_size(PARTITIONED_DIR),
}
for name, size in sizes.items():
    print(f"{name:22s}: {size / 1024 / 1024:7.2f} MiB")

compression_ratio = sizes["CSV"] / sizes["Parquet"]
print(f"\nКоэффициент C = S_CSV / S_Parquet = {compression_ratio:.2f}x")

assert sizes["Parquet"] < sizes["CSV"], "Parquet должен быть компактнее CSV"
assert compression_ratio > 1.0

CSV                   :   33.96 MiB
Parquet               :   22.68 MiB
Partitioned Parquet   :   22.85 MiB

Коэффициент C = S_CSV / S_Parquet = 1.50x


## Упражнение 2.2. Энтропия и сжимаемость

Коэффициент из предыдущего упражнения получился скромным. Причина — колонка `payload`:
это SHA-256, то есть практически случайная строка. Данные с высокой энтропией не сжимаются
принципиально: в них нет повторов, которые можно было бы закодировать короче.

Запишите **тот же набор без колонки `payload`** в CSV и в Parquet, посчитайте коэффициент
заново и сравните с предыдущим.

In [9]:
NOPAY_CSV = ROOT / "csv_nopayload"
NOPAY_PQ  = ROOT / "parquet_nopayload"

df_nopay = df.drop("payload")

df_nopay.write.mode("overwrite").option("header", True).csv(str(NOPAY_CSV))
df_nopay.write.mode("overwrite").option("compression", "snappy").parquet(str(NOPAY_PQ))

sizes_nopay = {"CSV": dir_size(NOPAY_CSV), "Parquet": dir_size(NOPAY_PQ)}
ratio_nopay = sizes_nopay["CSV"] / sizes_nopay["Parquet"]

print(f"Без payload: CSV {sizes_nopay['CSV']/1024/1024:.2f} MiB, "
      f"Parquet {sizes_nopay['Parquet']/1024/1024:.2f} MiB")
print(f"Коэффициент без payload: {ratio_nopay:.2f}x  (с payload было {compression_ratio:.2f}x)")

ANSWER_2_2 = """
Коэффициент вырос в несколько раз. Причина в энтропии данных.

Колонка payload — это SHA-256, то есть последовательность, статистически неотличимая от случайной.
Её энтропия близка к максимальной, повторов нет, словарное кодирование бесполезно, RLE бесполезно,
и Snappy тоже почти ничего не находит. При этом payload занимает 64 символа из примерно 110 в
строке CSV, то есть больше половины объёма. Эти байты одинаково плохо сжимаются в обоих форматах
и «размывают» выигрыш Parquet.

Остальные колонки устроены противоположным образом: robot_id принимает 12 значений, sensor — два,
temperature — 150. Низкая энтропия, много повторов, идеальный материал для dictionary encoding и
RLE. Как только высокоэнтропийная колонка убрана, преимущество колоночного формата проявляется
в полную силу.

Практический вывод: заявления вида «Parquet сжимает в 10 раз» бессмысленны без указания структуры
данных. Выигрыш определяется энтропией колонок, а не форматом сам по себе.
"""
print(ANSWER_2_2)

assert ratio_nopay > compression_ratio, "Без высокоэнтропийной колонки выигрыш должен вырасти"

Без payload: CSV 15.22 MiB, Parquet 3.61 MiB
Коэффициент без payload: 4.21x  (с payload было 1.50x)

Коэффициент вырос в несколько раз. Причина в энтропии данных.

Колонка payload — это SHA-256, то есть последовательность, статистически неотличимая от случайной.
Её энтропия близка к максимальной, повторов нет, словарное кодирование бесполезно, RLE бесполезно,
и Snappy тоже почти ничего не находит. При этом payload занимает 64 символа из примерно 110 в
строке CSV, то есть больше половины объёма. Эти байты одинаково плохо сжимаются в обоих форматах
и «размывают» выигрыш Parquet.

Остальные колонки устроены противоположным образом: robot_id принимает 12 значений, sensor — два,
temperature — 150. Низкая энтропия, много повторов, идеальный материал для dictionary encoding и
RLE. Как только высокоэнтропийная колонка убрана, преимущество колоночного формата проявляется
в полную силу.

Практический вывод: заявления вида «Parquet сжимает в 10 раз» бессмысленны без указания структуры
данных. Выи

## Упражнение 2.3. Сравнение кодеков

Запишите `df_nopay` в Parquet четырьмя кодеками: `uncompressed`, `snappy`, `gzip`, `zstd`.
Для каждого измерьте время записи и итоговый размер. Заполните словарь `codec_stats`
в формате `{кодек: (секунды, байты)}`.

In [10]:
codec_stats = {}

for codec in ["uncompressed", "snappy", "gzip", "zstd"]:
    out = ROOT / f"pq_{codec}"
    t0 = time.perf_counter()
    df_nopay.write.mode("overwrite").option("compression", codec).parquet(str(out))
    elapsed = time.perf_counter() - t0
    codec_stats[codec] = (elapsed, dir_size(out))

print(f"{'кодек':<14}{'время, с':>10}{'размер, MiB':>14}{'C к uncompressed':>20}")
base = codec_stats["uncompressed"][1]
for codec, (t, s) in codec_stats.items():
    print(f"{codec:<14}{t:>10.2f}{s/1024/1024:>14.2f}{base/s:>20.2f}")

assert set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"}
assert codec_stats["gzip"][1] < codec_stats["snappy"][1], "gzip должен сжимать сильнее snappy"

кодек           время, с   размер, MiB    C к uncompressed
uncompressed        2.80          5.09                1.00
snappy              1.12          3.61                1.41
gzip                1.05          2.70                1.89
zstd                0.71          2.61                1.95


**Вопрос 2.4.** Почему Snappy используется по умолчанию, хотя gzip сжимает заметно сильнее?
Ответьте письменно.

In [11]:
ANSWER_2_4 = """
Выбор кодека — это обмен процессорного времени на объём диска и сети, и Snappy стоит в точке,
удобной для аналитических нагрузок.

Snappy жмёт умеренно, но распаковывается очень быстро — сотни мегабайт в секунду на ядро. Файлы
Parquet читаются гораздо чаще, чем пишутся, поэтому цена распаковки платится многократно, и
экономить надо именно на ней. gzip даёт заметно меньший объём, но и сжатие, и распаковка стоят
существенно дороже: при аналитическом сканировании процессор становится узким местом раньше диска.

ZSTD — современный компромисс: сжимает почти как gzip, а распаковывается близко к Snappy, поэтому
в новых проектах его выбирают всё чаще. Uncompressed имеет смысл только для промежуточных данных
с очень коротким временем жизни.

Практическое правило: gzip оправдан для холодного архива, который пишут один раз и читают редко;
Snappy или ZSTD — для данных, по которым регулярно ходят запросы.
"""
print(ANSWER_2_4)


Выбор кодека — это обмен процессорного времени на объём диска и сети, и Snappy стоит в точке,
удобной для аналитических нагрузок.

Snappy жмёт умеренно, но распаковывается очень быстро — сотни мегабайт в секунду на ядро. Файлы
Parquet читаются гораздо чаще, чем пишутся, поэтому цена распаковки платится многократно, и
экономить надо именно на ней. gzip даёт заметно меньший объём, но и сжатие, и распаковка стоят
существенно дороже: при аналитическом сканировании процессор становится узким местом раньше диска.

ZSTD — современный компромисс: сжимает почти как gzip, а распаковывается близко к Snappy, поэтому
в новых проектах его выбирают всё чаще. Uncompressed имеет смысл только для промежуточных данных
с очень коротким временем жизни.

Практическое правило: gzip оправдан для холодного архива, который пишут один раз и читают редко;
Snappy или ZSTD — для данных, по которым регулярно ходят запросы.



---
# Часть 3. Бенчмарк чтения (2 балла)

## Упражнение 3.1. Функция измерения

Напишите функцию `benchmark_read(fmt, path, predicate=None, repeats=3, schema=None)`, которая:

1. `repeats` раз читает набор из `path` в формате `fmt` (`"csv"` или `"parquet"`);
2. при заданном `predicate` применяет фильтр;
3. выполняет действие `count()` — **без него из-за ленивости ничего не прочитается**;
4. возвращает кортеж `(медиана времени, число строк)`.

Для CSV используйте `option("header", True)`; параметр `schema` пока оставьте
необязательным — он понадобится в упражнении 3.3.

In [15]:
def benchmark_read(fmt, path, predicate=None, repeats=3, schema=None):
    """Возвращает (медианное время чтения, число строк)."""
    times, counts = [], []
    for _ in range(repeats):
        t0 = time.perf_counter()
        if fmt == "csv":
            reader = spark.read.option("header", True)
            if schema is not None:
                reader = reader.schema(schema)
            else:
                reader = reader.option("inferSchema", True)
            x = reader.csv(str(path))
        else:
            x = spark.read.parquet(str(path))
        if predicate is not None:
            x = x.filter(predicate)
        counts.append(x.count())
        times.append(time.perf_counter() - t0)
    return statistics.median(times), counts[-1]


t, n = benchmark_read("parquet", PARQUET_DIR, repeats=3)
print(f"Чтение всего Parquet: {t:.3f} с, строк {n}")
assert n == N

Чтение всего Parquet: 0.413 с, строк 300000


## Упражнение 3.2. Сравнение трёх версий

Выполните одинаковый запрос `robot_id == "robot-7"` к каждой из трёх версий набора, минимум
по три повтора. Рассчитайте ускорение относительно CSV.

In [16]:
predicate = F.col("robot_id") == "robot-7"

csv_t,  csv_n  = benchmark_read("csv",     CSV_DIR,         predicate)
pq_t,   pq_n   = benchmark_read("parquet", PARQUET_DIR,     predicate)
part_t, part_n = benchmark_read("parquet", PARTITIONED_DIR, predicate)

print(f"CSV                 : {csv_t:6.3f} с, строк {csv_n}")
print(f"Parquet             : {pq_t:6.3f} с, строк {pq_n}")
print(f"Partitioned Parquet : {part_t:6.3f} с, строк {part_n}")
print()
print(f"Speedup CSV -> Parquet             : {csv_t / pq_t:.2f}x")
print(f"Speedup CSV -> Partitioned Parquet : {csv_t / part_t:.2f}x")

assert csv_n == pq_n == part_n == N // 12, "Все три версии должны вернуть одинаковое число строк"

CSV                 :  2.385 с, строк 25000
Parquet             :  0.426 с, строк 25000
Partitioned Parquet :  0.757 с, строк 25000

Speedup CSV -> Parquet             : 5.60x
Speedup CSV -> Partitioned Parquet : 3.15x


## Упражнение 3.3. Честное сравнение

В упражнении 3.1 CSV читался с `inferSchema=True`. Это удобно, но нечестно по отношению к CSV:
чтобы вывести типы, Spark делает **дополнительный полный проход** по файлу. Получается, что
мы сравниваем Parquet с CSV, прочитанным дважды.

Повторите замер CSV с **явно заданной схемой** — возьмите её у уже прочитанного Parquet
(`spark.read.parquet(...).schema`) — и сравните три числа: CSV с выводом типов, CSV со схемой
и Parquet.

In [18]:
explicit_schema = spark.read.parquet(str(PARQUET_DIR)).schema

csv_schema_t, csv_schema_n = benchmark_read("csv", CSV_DIR, predicate, schema=explicit_schema)

print(f"CSV + inferSchema : {csv_t:6.3f} с")
print(f"CSV + явная схема : {csv_schema_t:6.3f} с")
print(f"Parquet           : {pq_t:6.3f} с")
print(f"\nЧестный speedup CSV -> Parquet: {csv_schema_t / pq_t:.2f}x")

ANSWER_3_3 = """
Выигрыш Parquet оказался меньше, чем думали: раньше считали, что формат сам по себе лучше, но на самом деле это было связано с тем, что данные нужно было проверять дважды. При работе с CSV данные можно прочитать за один раз, и это делает сравнение более честным.

Это важно, потому что если сравнивать так, что один вариант имеет худшие условия, то результаты будут неправильными. Если бы мы не заметили это, то могли бы ошибочно решить, что Parquet лучше из-за других причин, и это было бы неправильно.

Parquet всё же имеет свои плюсы: файлы занимают меньше места на диске, данные хранятся в виде бинарного кода (как картинки или музыка на компьютере), и это не требует анализа текста. Если нужно использовать не все данные, то Parquet становится ещё лучше благодаря тому, что он не проверяет все данные, а только те, что нужны.

Вывод: перед тем как публиковать результаты сравнения, нужно убедиться, что все способы работают в одинаковых условиях.
"""
print(ANSWER_3_3)

assert csv_schema_n == csv_n
assert csv_schema_t < csv_t, "Без inferSchema чтение CSV должно ускориться"

CSV + inferSchema :  2.385 с
CSV + явная схема :  1.385 с
Parquet           :  0.426 с

Честный speedup CSV -> Parquet: 3.25x

Выигрыш Parquet оказался меньше, чем думали: раньше считали, что формат сам по себе лучше, но на самом деле это было связано с тем, что данные нужно было проверять дважды. При работе с CSV данные можно прочитать за один раз, и это делает сравнение более честным.

Это важно, потому что если сравнивать так, что один вариант имеет худшие условия, то результаты будут неправильными. Если бы мы не заметили это, то могли бы ошибочно решить, что Parquet лучше из-за других причин, и это было бы неправильно.

Parquet всё же имеет свои плюсы: файлы занимают меньше места на диске, данные хранятся в виде бинарного кода (как картинки или музыка на компьютере), и это не требует анализа текста. Если нужно использовать не все данные, то Parquet становится ещё лучше благодаря тому, что он не проверяет все данные, а только те, что нужны.

Вывод: перед тем как публиковать результа

---
# Часть 4. Pruning и мелкие файлы (2 балла)

## Упражнение 4.1. Partition pruning

Выполните `explain(mode="formatted")` для запроса к partitioned Parquet с фильтром по
`robot_id` и найдите в плане строку `PartitionFilters`.

In [21]:
(
    spark.read.parquet(str(PARTITIONED_DIR))
    .filter(F.col("robot_id") == "robot-7")
    .select("robot_id", "temperature")
    .explain(mode="formatted")
)

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [temperature#1806, robot_id#1809]
Batched: true
Location: InMemoryFileIndex [file:/content/lab02_storage/parquet_partitioned]
PartitionFilters: [isnotnull(robot_id#1809), (robot_id#1809 = robot-7)]
ReadSchema: struct<temperature:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [temperature#1806, robot_id#1809]

(3) Project [codegen id : 1]
Output [2]: [robot_id#1809, temperature#1806]
Input [2]: [temperature#1806, robot_id#1809]




In [23]:
ANSWER_4_1 = """
TODO: что содержит PartitionFilters, чем он отличается от PushedFilters,
      и что означает ReadSchema в этом плане?
      PartitionFilters - это условие, которое говорит Spark читать данные только для робота с ID = robot-7. Spark проверяет это условие до того, как откроет файлы. Он смотрит только на один из двенадцати каталогов, игнорируя остальные одиннадцать. Это называется отсечением разделов (partition pruning).
PushedFilters работают по-другому: они содержат условия для обычных столбцов и применяются внутри чтения файлов Parquet. Эти фильтры не останавливают чтение файла, но помогают пропустить некоторые группы строк. В нашем случае таких фильтров нет, потому что все условия уже использованы для отсечения ненужных разделов.
ReadSchema показывает, какие столбцы будут прочитаны из файла. В нашем запросе мы используем только столбец с температурой (ID робота восстанавливается из имени каталога). Поэтому другие столбцы, такие как sensor, время, вибрация и полезная нагрузка, не считываются. Это называется отсечением столбцов (column pruning).
Таким образом, в плане запроса используются два механизма: отсечение разделов удаляет ненужные каталоги, а отсечение столбцов - ненужные столбцы.
"""
print(ANSWER_4_1)


TODO: что содержит PartitionFilters, чем он отличается от PushedFilters,
      и что означает ReadSchema в этом плане?
      PartitionFilters - это условие, которое говорит Spark читать данные только для робота с ID = robot-7. Spark проверяет это условие до того, как откроет файлы. Он смотрит только на один из двенадцати каталогов, игнорируя остальные одиннадцать. Это называется отсечением разделов (partition pruning).
PushedFilters работают по-другому: они содержат условия для обычных столбцов и применяются внутри чтения файлов Parquet. Эти фильтры не останавливают чтение файла, но помогают пропустить некоторые группы строк. В нашем случае таких фильтров нет, потому что все условия уже использованы для отсечения ненужных разделов.
ReadSchema показывает, какие столбцы будут прочитаны из файла. В нашем запросе мы используем только столбец с температурой (ID робота восстанавливается из имени каталога). Поэтому другие столбцы, такие как sensor, время, вибрация и полезная нагрузка, не счи

## Упражнение 4.2. Column pruning в цифрах

Сравните время двух запросов к обычному (непартиционированному) Parquet:

* агрегация **только** по `temperature`;
* агрегация по `temperature` **и** по тяжёлой высокоэнтропийной колонке `payload`.

Второй запрос вынуждает Spark поднять с диска колонку, которая занимает большую часть объёма
файла. Каждый запрос — минимум три повтора, берите медиану.

Дополнительно выведите `explain` обоих вариантов и сравните строки `ReadSchema`.

In [24]:
def timed(fn, repeats=3):
    """Медианное время выполнения fn()."""
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return statistics.median(ts)


def read_with_payload():
    return (spark.read.parquet(str(PARQUET_DIR))
            .agg(F.avg("temperature"), F.max("payload")).collect())


def read_one_column():
    return (spark.read.parquet(str(PARQUET_DIR))
            .agg(F.avg("temperature")).collect())


t_all = timed(read_with_payload)
t_one = timed(read_one_column)
print(f"temperature + payload: {t_all:.3f} с")
print(f"только temperature   : {t_one:.3f} с")
print(f"Column pruning дал ускорение в {t_all / t_one:.2f}x")

spark.read.parquet(str(PARQUET_DIR)).select("temperature").agg(F.avg("temperature")).explain()

ANSWER_4_2 = """
Время, за которое компьютер загружает данные, зависит от количества информации, которую ему нужно прочитать с жесткого диска. Когда вы делаете первый запрос, система должна получить максимум данных из колонки под названием "payload". Эта колонка занимает много места в файле, поэтому система считывает всю её целиком. Во втором запросе колонка "payload" не нужна, и система её не читает. Все данные в файле расположены подряд, и система просто игнорирует ненужные колонки.
Также важно понимать, что когда вы делаете запрос, система сама решает, какие данные ей нужны, даже если вы не указали это явно. Catalyst — это часть системы, которая помогает принимать такие решения. Поэтому добавление лишних команд select не ускорит работу. Однако, планирование запросов (партиционирование) лучше делать заранее.
Основное правило: не используйте команду SELECT *, так как это заставляет систему читать все колонки, даже если вам нужны только некоторые. На больших таблицах разница между запросом двух нужных колонок из пятидесяти и всех пятидесяти колонок будет значительной, потому что система потратит много времени на чтение ненужных данны
"""
print(ANSWER_4_2)

temperature + payload: 0.806 с
только temperature   : 0.420 с
Column pruning дал ускорение в 1.92x
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[avg(temperature#1970)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=1297]
      +- HashAggregate(keys=[], functions=[partial_avg(temperature#1970)])
         +- FileScan parquet [temperature#1970] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/lab02_storage/parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<temperature:double>



Время, за которое компьютер загружает данные, зависит от количества информации, которую ему нужно прочитать с жесткого диска. Когда вы делаете первый запрос, система должна получить максимум данных из колонки под названием "payload". Эта колонка занимает много места в файле, поэтому система считывает всю её целиком. Во втором запросе колонка "payload" не нужна, и система её не читает

## Упражнение 4.3. Проблема мелких файлов

Партиционирование выглядит бесплатным улучшением, но это не так. Запишите набор,
партиционированный по искусственному ключу высокой кардинальности `ts_bucket` (200 значений),
и сравните с партиционированием по `robot_id` (12 значений):

* число созданных файлов;
* суммарный объём;
* средний размер одного файла;
* время чтения всего набора.

In [25]:
MANY_DIR = ROOT / "parquet_many_partitions"

df_buckets = df_nopay.withColumn(
    "ts_bucket", ((F.col("ts_ms") / 100) % 200).cast("int")
)
df_buckets.write.mode("overwrite").partitionBy("ts_bucket").parquet(str(MANY_DIR))


def file_stats(path: Path):
    """Возвращает (число файлов .parquet, суммарный размер, средний размер)."""
    files = [p for p in path.rglob("*.parquet") if p.is_file()]
    total = sum(p.stat().st_size for p in files)
    return len(files), total, total / max(len(files), 1)


n_few,  s_few,  avg_few  = file_stats(PARTITIONED_DIR)
n_many, s_many, avg_many = file_stats(MANY_DIR)

print(f"{'вариант':<28}{'файлов':>9}{'объём, MiB':>13}{'средний файл, KiB':>20}")
print(f"{'partitionBy(robot_id) [12]':<28}{n_few:>9}{s_few/1024/1024:>13.2f}{avg_few/1024:>20.1f}")
print(f"{'partitionBy(ts_bucket) [200]':<28}{n_many:>9}{s_many/1024/1024:>13.2f}{avg_many/1024:>20.1f}")

t_few  = timed(lambda: spark.read.parquet(str(PARTITIONED_DIR)).count())
t_many = timed(lambda: spark.read.parquet(str(MANY_DIR)).count())
print(f"\nЧтение целиком: 12 партиций {t_few:.3f} с, 200 партиций {t_many:.3f} с")

assert n_many > n_few
assert avg_many < avg_few

вариант                        файлов   объём, MiB   средний файл, KiB
partitionBy(robot_id) [12]         24        22.68               967.6
partitionBy(ts_bucket) [200]      400         4.50                11.5

Чтение целиком: 12 партиций 0.704 с, 200 партиций 6.004 с


## Упражнение 4.4. Технический вывод

Ответьте письменно:

1. Почему слишком большое число мелких partition ухудшает эффективность хранения и чтения?
2. По какому признаку выбирать колонку для `partitionBy`?
3. Когда партиционирование по `robot_id` в нашей задаче окажется бесполезным?

In [26]:
ANSWER_4_4 = """
Маленькие файлы имеют свои минусы. Например, формат Parquet добавляет много лишней информации к каждому файлу.
В большом файле эта лишняя информация почти незаметна, но в маленьком файле она занимает много места, увеличивая его размер. Сжатие данных тоже работает хуже с маленькими файлами, потому что алгоритмы сжатия лучше всего работают на больших кусках одинаковых данных, которых нет в маленьких файлах.
Когда мы читаем данные, нам нужно открыть каждый файл, прочитать дополнительную информацию и спланировать, что делать дальше.
Если у нас много маленьких файлов, это занимает больше времени на управление, чем на обработку самих данных. В системе HDFS есть ещё одна проблема: вся информация о файлах хранится в памяти одного компьютера, и если у нас слишком много маленьких файлов, эта память может закончиться.
Если мы будем разбивать данные по идентификатору робота, это может быть неэффективно и даже вредно, особенно если запросы
охватывают всех роботов. В этом случае нам придётся прочитать все каталоги, что займёт много времени, и дополнительные затраты на разбиение данных останутся. Если один робот генерирует большую часть данных, его часть станет очень большой, а другие части — очень маленькими, что замедлит обработку.
Если количество роботов увеличится, например, с 12 до 50 000, это создаст слишком много разных значений, и тогда лучше разбивать данные по дате, а внутри каждой части использовать дополнительное разбиение.
"""
print(ANSWER_4_4)


Маленькие файлы имеют свои минусы. Например, формат Parquet добавляет много лишней информации к каждому файлу. 
В большом файле эта лишняя информация почти незаметна, но в маленьком файле она занимает много места, увеличивая его размер. Сжатие данных тоже работает хуже с маленькими файлами, потому что алгоритмы сжатия лучше всего работают на больших кусках одинаковых данных, которых нет в маленьких файлах.
Когда мы читаем данные, нам нужно открыть каждый файл, прочитать дополнительную информацию и спланировать, что делать дальше. 
Если у нас много маленьких файлов, это занимает больше времени на управление, чем на обработку самих данных. В системе HDFS есть ещё одна проблема: вся информация о файлах хранится в памяти одного компьютера, и если у нас слишком много маленьких файлов, эта память может закончиться.
Если мы будем разбивать данные по идентификатору робота, это может быть неэффективно и даже вредно, особенно если запросы 
охватывают всех роботов. В этом случае нам придётся про

---
# Часть 5. TEACH CARD (2 балла)

Заполните паспорт педагогической адаптации выполненной инженерной задачи. Это **обязательная**
часть работы.

**Требования к заполнению.** Формулировки конкретные. «Расскажу про Parquet» не засчитывается.
Нужны измеримый результат, критерий успеха и честно названные ограничения.

**Подсказка по содержанию.** Эксперимент «CSV против Parquet» удобен для школы тем, что результат
виден глазами: два каталога с одними и теми же данными занимают разный объём, и это можно
измерить без единой строки распределённого кода — хватит `du -h`. Аналогия для урока: если
нужна только температура, читается только «стопка температур», а не все листы журнала целиком.

| Поле | Содержание |
|---|---|
| **Название учебного проекта** | Parquet, партиционирование и бенчмаркинг |
| **Целевая аудитория** | 10–11 класс |
| **Исследовательский вопрос** | Почему файлы с одинаковыми данными занимают разный объём и открываются с разной скоростью? |
| **Источник данных** | Сгенерированные данные с роботов |
| **Инженерная концепция, сохраняемая без упрощения** | Колоночное хранение данных и чтение только нужных данных |
| **Что упрощается относительно университетской лабораторной** | Ученики работают браузере, а не в программе. Они пишут не сложный код, сравнивают параметры: размер папок, время выполнения и простые расчёты |
| **Алгоритм действий обучающегося (4–6 шагов)** | Сгенерировать данные и сохранить их в форматах CSV и Parquet, Измерить и сравнить размер полученных папок, удалить из данных столбец со случайными значениями (хеш) и повторить замер размера, замерить время чтения только столбца «температура» из обоих форматов, рассчитать коэффициенты сжатия и ускорения, записать вывод |
| **Измеримый результат / метрика** | Коэфицент сжатия, ускорение чтения |
| **Критерий успешного выполнения** | Замеры подтверждают: parquet компактнее и быстрее csv Сравнивение: читать один столбец, а не всю строку |
| **Вариант усложнения для НТО / хакатона** | Подобрать кодек сжатия для баланса между размером файла и скоростью чтения. |
| **Риски и ограничения** | Файлы parquet невозможно просмотреть в обычном текстовом редакторе |

---
## Итоговая самопроверка

In [ ]:
checks = {
    "1.1 CSV записан":          CSV_DIR.exists(),
    "1.2 Parquet записан":      len(list(PARQUET_DIR.glob("*.parquet"))) > 0,
    "1.3 partitionBy":          len(part_dirs) == 12,
    "1.4 схема партиции":       "TODO" not in ANSWER_1_4,
    "2.1 размеры и C":          compression_ratio > 1.0,
    "2.2 энтропия":             ratio_nopay > compression_ratio and "TODO" not in ANSWER_2_2,
    "2.3 кодеки":               set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"},
    "2.4 выбор кодека":         "TODO" not in ANSWER_2_4,
    "3.2 бенчмарк чтения":      csv_n == pq_n == part_n,
    "3.3 честное сравнение":    csv_schema_t < csv_t and "TODO" not in ANSWER_3_3,
    "4.1 partition pruning":    "TODO" not in ANSWER_4_1,
    "4.2 column pruning":       t_all > t_one and "TODO" not in ANSWER_4_2,
    "4.3 мелкие файлы":         n_many > n_few,
    "4.4 технический вывод":    "TODO" not in ANSWER_4_4,
}

for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")

print()
print(f"Выполнено: {sum(checks.values())} из {len(checks)}")
print("TEACH CARD проверяется преподавателем вручную.")

In [ ]:
df.unpersist()
spark.stop()
print("SparkSession остановлена.")